In [1]:
import pandas as pd
%pip install -U nltk
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
import re
import pandas as pd
import os, ssl


Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to C:\Users\Garance
[nltk_data]     Latieule/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# -*- coding: utf-8 -*-
# Parser for ECB speeches (//-separated) and conferences (comma CSV)
# Outputs: Doc_ID, Speaker, Date, contents, Parsed_Text, Sentence_Rank

import os
import ssl
import re
from typing import List, Optional
import pandas as pd
import nltk

# -----------------------------
# Input paths (edit as needed)
# -----------------------------
path_speech = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\all_ECB_speeches.csv"      # //-separated
path_conf   = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Scraping ECB conferences\all_ECB_conferences.csv"   # comma CSV

# Optional outputs (set to None to skip saving)
out_speech  = None  # e.g., r"C:\...\all_ECB_speeches_parsed.csv"
out_conf    = None  # e.g., r"C:\...\all_ECB_conferences_parsed.csv"
SAVE_WITH_DOUBLE_SLASH = True   # if saving, write outputs with // separator

# -----------------------------
# Helper: // writer
# -----------------------------
def write_double_slash_csv(df: pd.DataFrame, output_file: str, order: Optional[List[str]] = None) -> None:
    """Write a `//`-separated text file with UTF-8 BOM. Escapes '//' inside values to '\/\//'."""
    if order:
        lower_map = {c.lower(): c for c in df.columns}
        missing = [c for c in order if c.lower() not in lower_map]
        if missing:
            raise KeyError(f"Missing columns: {missing}. Available: {list(df.columns)}")
        cols = [lower_map[c.lower()] for c in order]
        df = df[cols]
    else:
        cols = list(df.columns)

    def sanitize(val) -> str:
        if pd.isna(val):
            return ""
        if isinstance(val, pd.Timestamp):
            s = val.strftime("%Y-%m-%d")
        else:
            s = str(val)
        s = s.replace("\r\n", " ").replace("\r", " ").replace("\n", " ")
        s = s.replace("//", r"\/\//")
        return s

    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    header = "//".join(cols)
    with open(output_file, "w", encoding="utf-8-sig", newline="") as f:
        f.write(header + "\n")
        for _, row in df.iterrows():
            f.write("//".join(sanitize(row[c]) for c in cols) + "\n")

# -----------------------------
# NLTK setup (local dir + SSL)
# -----------------------------
NLTK_DIR = os.path.join(os.getcwd(), ".nltk_data_ecb")
os.makedirs(NLTK_DIR, exist_ok=True)
if NLTK_DIR not in nltk.data.path:
    nltk.data.path.append(NLTK_DIR)

try:
    _create_unverified_https_context = ssl._create_unverified_context
    ssl._create_default_https_context = _create_unverified_https_context
except Exception:
    pass

def _ensure_resource(res_name: str, pkg_override: Optional[str] = None) -> bool:
    """Ensure an NLTK resource exists; download quietly to local dir if needed."""
    try:
        nltk.data.find(res_name)
        return True
    except LookupError:
        try:
            pkg = pkg_override or res_name.split("/")[1]
            nltk.download(pkg, download_dir=NLTK_DIR, quiet=True)
            nltk.data.find(res_name)
            return True
        except Exception:
            return False

has_punkt = _ensure_resource("tokenizers/punkt")
_ = _ensure_resource("tokenizers/punkt_tab", pkg_override="punkt_tab")  # some installs need this

# -----------------------------
# Tokenizer (NLTK or fallback)
# -----------------------------
if has_punkt:
    from nltk.tokenize import sent_tokenize as _nltk_sent_tokenize
    def safe_sent_tokenize(text: str) -> List[str]:
        if not isinstance(text, str):
            text = "" if text is None else str(text)
        return [s for s in _nltk_sent_tokenize(text) if s.strip()]
else:
    # Simple heuristic: split on sentence end followed by uppercase letter
    _SPLIT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-ZÉÈÀÂÎÏÔÙÜÇ])')
    def safe_sent_tokenize(text: str) -> List[str]:
        if not isinstance(text, str):
            text = "" if text is None else str(text)
        parts = _SPLIT_RE.split(text)
        return [p.strip() for p in parts if p and p.strip()]

# -----------------------------
# Speaker extraction (for conferences)
# -----------------------------
_GENERIC_PAT = re.compile(r"\b(ecb|press conference|monetary policy statement)\b", flags=re.I)
def extract_speaker_from_title(title: str, ignore_generics: bool = True) -> str:
    """Take substring before the first ':' as speaker. Optionally ignore generic prefixes."""
    if not isinstance(title, str):
        return ""
    i = title.find(":")
    if i <= 0:
        return ""
    candidate = title[:i].strip()
    if ignore_generics and _GENERIC_PAT.search(candidate):
        return ""
    return candidate

# -----------------------------
# Processor: SPEECHES (// file)
# -----------------------------
def process_speeches(path: str) -> pd.DataFrame:
    """
    Load //-separated 'all_ECB_speeches.csv' with columns:
    date, speakers, title, subtitle, contents
    Return normalized parsed rows.
    """
    df = pd.read_csv(
        path,
        sep="//",
        engine="python",          # required for // separator
        encoding="utf-8-sig",
        on_bad_lines="warn"
    )

    # Validate columns (case-insensitive)
    lower_map = {c.lower(): c for c in df.columns}
    required = ["date", "speakers", "title", "subtitle", "contents"]
    missing = [c for c in required if c not in lower_map]
    if missing:
        raise KeyError(f"[{os.path.basename(path)}] Missing columns: {missing}. Available: {list(df.columns)}")

    # Canonical names
    df = df.rename(columns={
        lower_map["date"]: "date",
        lower_map["speakers"]: "speakers",
        lower_map["title"]: "title",
        lower_map["subtitle"]: "subtitle",
        lower_map["contents"]: "contents",
    })

    # Doc_ID (1-based)
    df.insert(0, "doc_id", range(1, len(df) + 1))

    # Build base frame:
    # - Speaker comes directly from 'speakers'
    # - contents is already 'contents'
    tmp = df[["doc_id", "date", "speakers", "contents"]].copy()
    tmp["Speaker"] = tmp["speakers"].fillna("").astype(str).str.strip()
    tmp["contents"] = tmp["contents"].fillna("").astype(str)
    tmp["Date"] = pd.to_datetime(tmp["date"], errors="coerce").dt.strftime("%Y-%m-%d").fillna(tmp["date"])

    # Tokenize
    tmp["Parsed_Text"] = tmp["contents"].apply(safe_sent_tokenize)
    parsed = tmp.explode("Parsed_Text", ignore_index=True)
    parsed = parsed[parsed["Parsed_Text"].astype(str).str.strip().ne("")]

    # Final shape + rank
    parsed = parsed.rename(columns={"doc_id": "Doc_ID"})[["Doc_ID", "Speaker", "Date", "contents", "Parsed_Text"]]
    parsed["Sentence_Rank"] = parsed.groupby("Doc_ID").cumcount()

    return parsed.reset_index(drop=True)

# -----------------------------
# Processor: CONFERENCES (, CSV)
# -----------------------------
def process_conferences(path: str, infer_speaker: bool = True) -> pd.DataFrame:
    """
    Load comma CSV 'all_ECB_conferences.csv' with columns:
    date, title, link, text
    Return normalized parsed rows.
    """
    df = pd.read_csv(
        path,
        encoding="utf-8-sig",
        on_bad_lines="warn"
    )

    lower_map = {c.lower(): c for c in df.columns}
    required = ["date", "title", "link", "text"]
    missing = [c for c in required if c not in lower_map]
    if missing:
        raise KeyError(f"[{os.path.basename(path)}] Missing columns: {missing}. Available: {list(df.columns)}")

    # Canonical names
    df = df.rename(columns={
        lower_map["date"]: "date",
        lower_map["title"]: "title",
        lower_map["link"]: "link",
        lower_map["text"]: "text",
    })

    # Doc_ID (1-based)
    df.insert(0, "doc_id", range(1, len(df) + 1))

    tmp = df[["doc_id", "date", "title", "text"]].copy()
    tmp["contents"] = tmp["text"].fillna("").astype(str)
    tmp["Date"] = pd.to_datetime(tmp["date"], errors="coerce").dt.strftime("%Y-%m-%d").fillna(tmp["date"])

    # Speaker: infer from title (or set empty if not needed)
    if infer_speaker:
        tmp["Speaker"] = tmp["title"].apply(lambda t: extract_speaker_from_title(t, ignore_generics=True))
    else:
        tmp["Speaker"] = ""

    # Tokenize
    tmp["Parsed_Text"] = tmp["contents"].apply(safe_sent_tokenize)
    parsed = tmp.explode("Parsed_Text", ignore_index=True)
    parsed = parsed[parsed["Parsed_Text"].astype(str).str.strip().ne("")]

    parsed = parsed.rename(columns={"doc_id": "Doc_ID"})[["Doc_ID", "Speaker", "Date", "contents", "Parsed_Text"]]
    parsed["Sentence_Rank"] = parsed.groupby("Doc_ID").cumcount()

    return parsed.reset_index(drop=True)

# -----------------------------
# Run both
# -----------------------------
new_df_parsed_speech = process_speeches(path_speech)
new_df_parsed_conf   = process_conferences(path_conf, infer_speaker=True)

print(f"[OK] Speeches:    {new_df_parsed_speech['Doc_ID'].nunique()} docs -> {len(new_df_parsed_speech):,} sentences.")
print(f"[OK] Conferences: {new_df_parsed_conf['Doc_ID'].nunique()} docs -> {len(new_df_parsed_conf):,} sentences.")

# -----------------------------
# Optional save
# -----------------------------
if out_speech:
    if SAVE_WITH_DOUBLE_SLASH:
        write_double_slash_csv(
            new_df_parsed_speech,
            out_speech,
            order=["Doc_ID", "Speaker", "Date", "contents", "Parsed_Text", "Sentence_Rank"]
        )
    else:
        new_df_parsed_speech.to_csv(out_speech, index=False, encoding="utf-8-sig")

if out_conf:
    if SAVE_WITH_DOUBLE_SLASH:
        write_double_slash_csv(
            new_df_parsed_conf,
            out_conf,
            order=["Doc_ID", "Speaker", "Date", "contents", "Parsed_Text", "Sentence_Rank"]
        )
    else:
        new_df_parsed_conf.to_csv(out_conf, index=False, encoding="utf-8-sig")


<>:27: SyntaxWarning: invalid escape sequence '\/'
<>:27: SyntaxWarning: invalid escape sequence '\/'
C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_86068\3635656002.py:27: SyntaxWarning: invalid escape sequence '\/'
  """Write a `//`-separated text file with UTF-8 BOM. Escapes '//' inside values to '\/\//'."""
C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_86068\3635656002.py:130: ParserWarning: Skipping line 275: Expected 5 fields in line 275, saw 6. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  df = pd.read_csv(
C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_86068\3635656002.py:130: ParserWarning: Skipping line 389: Expected 5 fields in line 389, saw 6. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  df = pd.read_csv(
C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_86068\3635656002.py:130: ParserWarning: Skipping line 405: Expected 5 fields in line 405, saw 6. Err

[OK] Speeches:    2681 docs -> 327,241 sentences.
[OK] Conferences: 326 docs -> 72,400 sentences.


In [ ]:
# -----------------------------
# Continue: clean both datasets
# -----------------------------
import re
import pandas as pd

def clean_parsed_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply the requested text cleaning steps on a parsed dataframe that contains:
    ['Doc_ID', 'Speaker', 'Date', 'contents', 'Parsed_Text', 'Sentence_Rank']
    """
    df = df.copy()

    # 1) Neutralize NaN and force string
    df["Parsed_Text"] = df["Parsed_Text"].fillna("").astype(str)

    # 2) Remove punctuation (keep letters/digits/spaces)
    df["Parsed_Text"] = df["Parsed_Text"].str.replace(r"[^\w\s]", "", regex=True)

    # 3) Normalize whitespaces
    df["Parsed_Text"] = df["Parsed_Text"].str.replace(r"\s+", " ", regex=True).str.strip()

    # 4) Filter out very short sentences
    df = df[df["Parsed_Text"].str.len() >= 20]

    # 5) Remove common redundant strings (exactly as provided)
    repl_patterns = [
        (r"cid173", ""),
        (r"digitized for fraser httpfraserstlouisfedorg federal reserve bank of st louis", ""),
        (r"UFB03", "ffi"),
        (r"U0080U0099", ""),
        (r"U0080U0091", ""),
        (r"U0080U0094", ""),
        (r"U0080", ""),
        (r"U0099", ""),
        (r"U0093", ""),
        (r"U009", ""),  # kept as requested, even if broad
    ]
    for pat, repl in repl_patterns:
        df["Parsed_Text"] = df["Parsed_Text"].apply(lambda x: re.sub(pat, repl, x))

    # 6) Lowercase
    df["Parsed_Text"] = df["Parsed_Text"].map(str.lower)

    # 7) Drop duplicate sentences globally (exactly as requested)
    df = df.drop_duplicates(subset=["Parsed_Text"]).reset_index(drop=True)

    # Keep column order consistent
    wanted_order = ["Doc_ID", "Speaker", "Date", "contents", "Parsed_Text", "Sentence_Rank"]
    existing = [c for c in wanted_order if c in df.columns]
    rest = [c for c in df.columns if c not in existing]
    df = df[existing + rest]

    return df

# Apply to both datasets
df_speech_clean = clean_parsed_df(new_df_parsed_speech)
df_conf_clean   = clean_parsed_df(new_df_parsed_conf)

# delete the column 'contents' (otherwise the file is too heavy for git push) and save the intermediate file
df_speech_clean  = df_speech_clean.drop(columns=['contents'], errors='ignore')
df_conf_clean  = df_conf_clean.drop(columns=['contents'], errors='ignore')

out_path_speech = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_speech_intermediate.csv"
out_path_conf = r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_conf_intermediate.csv"
df_speech_clean.to_csv(out_path_speech, index=False, encoding="utf-8-sig")
df_conf_clean.to_csv(out_path_conf, index=False, encoding="utf-8-sig")

print(f"[OK] Saved {len(df_speech_clean):,} rows -> {out_path_speech}")
print(f"[OK] Saved {len(df_conf_clean):,} rows -> {out_path_conf}")



[OK] Saved 306,153 rows -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_speech_intermediate.csv
[OK] Saved 67,505 rows -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_conf_intermediate.csv


: 

pb qd texte pas anglais

In [ ]:
# --- Détection (langid) + traduction vers l'anglais pour 2 CSV ---
# (à coller dans une cellule Jupyter)

# Si besoin d'installer:
# %pip install -U deep-translator
# %pip install -U deep_translator

import os, time
import pandas as pd
import langid
from deep_translator import GoogleTranslator
from tqdm import tqdm

# ================== PARAMS ==================
TEXT_COL     = "Parsed_Text"       # colonne texte source
OUT_LANG_COL = "lang"              # colonne code langue détectée
OUT_TEXT_EN  = "Parsed_Text_en"    # texte traduit (ou original si en/und)

# Les deux fichiers à traiter
JOBS = [
    r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_conf_intermediate.csv",
    r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_speech_intermediate.csv",
]

# Ordre de colonnes préféré (si présent)
PREFERRED = ["Doc_ID", "Speaker", "Date", "contents", "Parsed_Text", "Sentence_Rank"]

# ================== UTILS ==================
def safe_detect(text: str) -> str:
    s = "" if pd.isna(text) else str(text).strip()
    if not s:
        return "und"
    try:
        code, _ = langid.classify(s)
        return code or "und"
    except Exception:
        return "und"

def translate_batch(texts, langs, translator, cache: dict, retries=3, sleep_s=0.5):
    """Traduit vers EN uniquement si langue != 'en' et != 'und'.
       Utilise un petit cache et des retries simples."""
    out = []
    for txt, lg in tqdm(zip(texts, langs), total=len(texts), desc="Traduction -> EN"):
        s = "" if pd.isna(txt) else str(txt).strip()
        if not s or lg in {"en", "und"}:
            out.append(txt)
            continue
        key = (s, lg)
        if key in cache:
            out.append(cache[key])
            continue
        translated = txt
        for attempt in range(retries):
            try:
                translated = translator.translate(s)
                break
            except Exception:
                # quota/réseau : petite pause puis retry
                time.sleep(sleep_s * (attempt + 1))
        cache[key] = translated
        out.append(translated)
    return out

# ================== MAIN ==================
for in_path in JOBS:
    assert os.path.isfile(in_path), f"Introuvable: {in_path}"
    print(f"\n=== Traitement: {in_path} ===")

    # 1) Lire
    df = pd.read_csv(in_path)
    if TEXT_COL not in df.columns:
        raise ValueError(f"Colonne '{TEXT_COL}' introuvable. Colonnes: {list(df.columns)}")

    # 2) Détection langue
    tqdm.pandas(desc="Détection de langue (langid)")
    df[OUT_LANG_COL] = df[TEXT_COL].progress_apply(safe_detect)

    # 3) Traduction conditionnelle -> EN
    translator = GoogleTranslator(source="auto", target="en")
    _cache = {}
    df[OUT_TEXT_EN] = translate_batch(
        texts=df[TEXT_COL].tolist(),
        langs=df[OUT_LANG_COL].tolist(),
        translator=translator,
        cache=_cache
    )

    # 4) Sauvegarde (ordre préféré + nouvelles colonnes en fin)
    out_dir = os.path.dirname(in_path)
    base = os.path.splitext(os.path.basename(in_path))[0]
    out_path_en = os.path.join(out_dir, f"{base}_en.csv")

    ordered = [c for c in PREFERRED if c in df.columns]
    tail = [c for c in df.columns if c not in ordered and c not in {OUT_LANG_COL, OUT_TEXT_EN}]
    final_cols = ordered + tail + [OUT_LANG_COL, OUT_TEXT_EN]

    df.to_csv(out_path_en, index=False, encoding="utf-8-sig", columns=final_cols)
    print(f"[OK] Détecté (langid) & traduit si besoin : {len(df):,} lignes -> {out_path_en}")

#Time ~~ 10 minutes


=== Traitement: C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_conf_intermediate.csv ===


Traduction -> EN: 100%|██████████| 67505/67505 [02:30<00:00, 448.73it/s] 


[OK] Détecté (langid) & traduit si besoin : 67,505 lignes -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_conf_intermediate_en.csv

=== Traitement: C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_speech_intermediate.csv ===


Traduction -> EN:  48%|████▊     | 147409/306153 [8:48:41<7992:10:04, 181.25s/it] 

In [ ]:
# %% [markdown]
# One-cell cleaner (stdlib only): stopwords + boilerplate + months/days + artifacts
# Protects economic vocabulary & negations; optional simple lemmatization; streams CSV.

import csv, re
from typing import Set, Optional, Iterable

# ===================== 1) PARAMÈTRES GÉNÉRAUX =====================
TEXT_COL = "Parsed_Text"       # colonne texte; None = auto-détection (Parsed_Text/contents/text/sentence)
NEW_COL  = "Parsed_Text_clean" # nouvelle colonne nettoyée
KEEP_NUMBERS = True            # garder les nombres
USE_LEMMA    = True            # lemmatisation simple

# Deux jeux de fichiers à traiter
JOBS = [
    (
        r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_conf_intermediate.csv",
        r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Final outputs\pre_processed_ECB_conf_final.csv",
    ),
    (
        r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_speech_intermediate.csv",
        r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Final outputs\pre_processed_ECB_speech_final.csv",
    ),
]

# ===================== 2) STOPLIST =====================
BASE_STOP = {
    # Articles / déterminants
    "the","a","an","some","any","each","every","either","neither","another","other","others",
    "such","this","that","these","those","same","own","former","latter",

    # Pronoms (pers., poss., démonstratifs, relatifs, indéfinis)
    "i","me","my","mine","myself","we","us","our","ours","ourselves",
    "you","your","yours","yourself","yourselves",
    "he","him","his","himself","she","her","hers","herself",
    "it","its","itself","they","them","their","theirs","themselves",
    "one","ones","someone","somebody","anyone","anybody","everyone","everybody",
    "noone","nobody","whose","which","who","whom","where","when","why","how",
    "whatever","whichever","whoever","whomever", "what",

    # Prépositions / particules
    "of","in","on","at","by","for","with","as","from","to","into","onto","over","under",
    "above","below","between","among","through","throughout","within","without","via",
    "across","along","around","behind","beyond","before","after","during","outside","inside",
    "about","regarding","concerning","per","versus","vs","amid","amidst","upon","off",

    # Conjonctions / connecteurs
    "and","or","but","so","than","then","that","if","though","although","because","since",
    "unless","until","while","whereas","whether","thus","therefore","hence","whereby",
    "thereby","therein","thereof","hereby","herein","furthermore","moreover","additionally",
    "meanwhile","nonetheless","nevertheless","instead","otherwise","likewise","similarly",

    # Adverbes fréquents / intensifieurs / fillers
    "very","too","also","just","still","only","even","ever","yet","again","almost","nearly",
    "roughly","around","approximately","about","rather","quite","pretty","fairly","truly",
    "indeed","simply","clearly","obviously","apparently","basically","essentially","largely",
    "generally","typically","commonly","mostly","mainly","primarily","overall","anyway","already",

    # Auxiliaires (BE / HAVE / DO) — (les négations seront normalisées en 'not')
    "be","am","is","are","was","were","been","being",
    "have","has","had","having",
    "do","does","did","doing",

    # Modaux
    "can","could","may","might","shall","should","will","would","must","ought",

    # Verbes de rapport / discours (peu informatifs)
    "say","says","said","saying",
    "tell","tells","told","telling",
    "state","states","stated","stating",
    "note","notes","noted","noting",
    "remark","remarks","remarked","remarking",
    "mention","mentions","mentioned","mentioning",
    "discuss","discusses","discussed","discussing",
    "address","addresses","addressed","addressing",
    "announce","announces","announced","announcing",
    "report","reports","reported","reporting",
    "explain","explains","explained","explaining",
    "highlight","highlights","highlighted","highlighting",
    "outline","outlines","outlined","outlining",
    "emphasize","emphasizes","emphasized","emphasizing",
    "point","points","pointed","pointing",
    "argue","argues","argued","arguing",
    "believe","believes","believed","believing",
    "think","thinks","thought","thinking",
    "consider","considers","considered","considering",
    "estimate","estimates","estimated","estimating",
    "assume","assumes","assumed","assuming",
    "suggest","suggests","suggested","suggesting",
    "propose","proposes","proposed","proposing",
    "acknowledge","acknowledges","acknowledged","acknowledging",

    # Verbes "légers" / génériques
    "make","makes","made","making",
    "take","takes","took","taking",
    "give","gives","gave","giving",
    "get","gets","got","getting",
    "put","puts","put","putting",
    "keep","keeps","kept","keeping",
    "use","uses","used","using",
    "provide","provides","provided","providing",
    "include","includes","included","including",
    "allow","allows","allowed","allowing",
    "enable","enables","enabled","enabling",
    "maintain","maintains","maintained","maintaining",
    "enhance","enhances","enhanced","enhancing",
    "deliver","delivers","delivered","delivering",
    "drive","drives","drove","driving",
    "lead","leads","led","leading",
    "bring","brings","brought","bringing",
    "show","shows","showed","showing",
    "look","looks","looked","looking",
    "work","works","worked","working",
    "go","goes","went","going",
    "come","comes","came","coming",
    "need","needs","needed","needing",
    "want","wants","wanted","wanting",
    "try","tries","tried","trying",
    "aim","aims","aimed","aiming",
    "intend","intends","intended","intending",
    "appear","appears","appeared","appearing",
    "seem","seems","seemed","seeming",

    # Politesse / méta-discours
    "welcome","address","opening","closing","conclusion","thank","thanks","thanking",
    "good","morning","afternoon","evening","today","tonight","everyone","folks",
    "ladies","gentlemen","colleagues","friends","remarks","keynote",
    "conference","symposium","panel","session","event","meeting","host","hosted",
    "speech","speeches","statement","statements","introductory","press","release","blog","post",
    "member","members","chair","chairman","chairwoman","governor","president",
    "vice","deputy","board","executive","committee","department","division",

    # Mois / jours / temps génériques
    "january","february","march","april","may","june","july","august","september",
    "october","november","december",
    "monday","tuesday","wednesday","thursday","friday","saturday","sunday",
    "today","yesterday","tomorrow","tonight","morning","afternoon","evening",
    "week","weeks","month","months","year","years","quarter","quarters","semester","semesters",
    "day","days","daily","monthly","yearly","annually","quarterly",

    # Quantificateurs / mesure vagues
    "many","much","few","several","plenty","various","numerous","multiple",
    "more","most","less","least","greater","greatest","smaller","smallest",
    "somewhat","kind","sort","type","kinds","sorts","types",

    # Deixis / locatifs génériques
    "here","there","where","anywhere","everywhere","somewhere","nowhere","near","far",

    #divers
    "welcome","address","opening","closing","conclusion","thank","thanks","thanking",
    "good","morning","afternoon","evening","today","tonight","everyone",
    "ladies","gentlemen","colleagues","friends","remarks","keynote",
    "conference","symposium","panel","session","event","meeting","host","hosted",
    "speech","speeches","statement","statements","introductory","press","release","blog","post",
    "member","members","chair","chairman","chairwoman","governor","president",
    "vice","deputy","board","executive","committee","department","division",
    "let","me","us","want","like","say","ask","think","know","believe","going",
    "talk","speak","turn","point","points","make","made","making",
    "slide","slides","download","click","page","pages","copyright","video","pdf","transcript",
    "press","release","introductory","blog","post"
}

FINAL_STOP: Set[str] = BASE_STOP

# ===================== 3) NETTOYAGE =====================
TOKEN_RE = re.compile(r"[a-z0-9]+")

def simple_lemma(w: str) -> str:
    if len(w) <= 3: return w
    if w.endswith("ing") and len(w) > 5:
        base = w[:-3]
        if base.endswith(("iz","at")): return base + "e"
        if len(base) >= 2 and base[-1] == base[-2]: base = base[:-1]
        return base
    if w.endswith("ied") and len(w) > 4: return w[:-3] + "y"
    if w.endswith("ed") and len(w) > 4:
        base = w[:-2]
        if base.endswith(("iz","at")): return base + "e"
        if len(base) >= 2 and base[-1] == base[-2]: base = base[:-1]
        return base
    if w.endswith("ies") and len(w) > 4: return w[:-3] + "y"
    if w.endswith("es") and len(w) > 3 and w[-3] in ("x","s","z","h"): return w[:-2]
    if w.endswith("s") and len(w) > 3 and not w.endswith("ss"): return w[:-1]
    return w

def clean_sentence(text: str, stopset: Set[str]=FINAL_STOP, keep_numbers: bool=True, use_lemma: bool=True) -> str:
    if not isinstance(text, str): text = "" if text is None else str(text)
    toks = TOKEN_RE.findall(text.lower())
    if not keep_numbers: toks = [t for t in toks if not t.isdigit()]
    toks = [t for t in toks if t not in stopset]
    if use_lemma: toks = [simple_lemma(t) for t in toks]
    toks = [t for t in toks if t]
    return " ".join(toks)

def build_fields_out(original_fields: Iterable[str], desired_order: Iterable[str], new_col: str) -> list:
    """Construit l’ordre des colonnes: desired_order d’abord (si présentes),
    puis le reste dans l’ordre d’origine. Insert NEW_COL juste après Parsed_Text si possible."""
    original_fields = list(original_fields)
    ordered = [c for c in desired_order if c in original_fields]
    for c in original_fields:
        if c not in ordered:
            ordered.append(c)
    if new_col not in ordered:
        if "Parsed_Text" in ordered:
            idx = ordered.index("Parsed_Text") + 1
            ordered.insert(idx, new_col)
        else:
            ordered.append(new_col)
    return ordered

def process_csv(in_csv: str, out_csv: str, text_col: Optional[str]="Parsed_Text", new_col: str="Parsed_Text_clean",
                keep_numbers: bool=True, use_lemma: bool=True) -> int:
    """
    Lit in_csv, crée new_col, et ÉCRIT out_csv en CONSERVANT la colonne text_col (Parsed_Text).
    Respecte l’ordre préféré si possible: [Doc_ID, Speaker, Date, contents, Parsed_Text, Sentence_Rank] + new_col.
    """
    n = 0
    with open(in_csv, "r", newline="", encoding="utf-8") as fin:
        reader = csv.DictReader(fin)
        fields = list(reader.fieldnames or [])
        if not fields:
            print("[Erreur] CSV sans en-tête."); 
            return 0

        # auto-détection éventuelle de la colonne texte
        if text_col is None or text_col not in fields:
            for cand in ("Parsed_Text","contents","text","sentence"):
                if cand in fields: text_col = cand; break
        if text_col not in fields:
            print(f"[Erreur] Colonne texte introuvable. Dispo: {fields}")
            return 0

        # ordre préféré (si les colonnes existent)
        PREFERRED = ["Doc_ID","Speaker","Date","contents","Parsed_Text","Sentence_Rank"]
        fields_out = build_fields_out(fields, PREFERRED, new_col)

        with open(out_csv, "w", newline="", encoding="utf-8") as fout:
            writer = csv.DictWriter(fout, fieldnames=fields_out, extrasaction="ignore")
            writer.writeheader()
            for row in reader:
                raw = row.get(text_col, "") or ""
                row[new_col] = clean_sentence(raw, stopset=FINAL_STOP,
                                              keep_numbers=keep_numbers, use_lemma=use_lemma)
                writer.writerow(row)
                n += 1
                if n % 100000 == 0:
                    print(f"[Info] {n} lignes traitées pour {in_csv}...")
    return n

# ===================== 4) RUN: deux fichiers =====================
total = 0
for IN_CSV, OUT_CSV in JOBS:
    n_lines = process_csv(
        IN_CSV, OUT_CSV, text_col=TEXT_COL, new_col=NEW_COL,
        keep_numbers=KEEP_NUMBERS, use_lemma=USE_LEMMA
    )
    print(f"[OK] {n_lines} lignes traitées. Écrit -> {OUT_CSV}")
    total += n_lines

print(f"[DONE] Total lignes traitées (2 fichiers) : {total}")

# Aperçu rapide (si pandas dispo)
try:
    import pandas as pd
    for _, OUT in JOBS:
        print(f"\nAperçu: {OUT}")
        df = pd.read_csv(OUT, nrows=5)
        cols = [c for c in ["Doc_ID","Speaker","Date","contents","Parsed_Text","Sentence_Rank", NEW_COL] if c in df.columns]
        display(df[cols] if cols else df.head())
except Exception:
    pass


[OK] 67505 lignes traitées. Écrit -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Final outputs\pre_processed_ECB_conf_final.csv
[Info] 100000 lignes traitées pour C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_speech_intermediate.csv...
[Info] 200000 lignes traitées pour C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_speech_intermediate.csv...
[Info] 300000 lignes traitées pour C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Intermediate outputs\pre_processed_ECB_speech_intermediate.csv...
[OK] 306153 lignes traitées. Écrit -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Final outputs\pre_processed_ECB_speech_final.csv
[DONE] Total lignes traitées (2 fichiers) : 373658

Aperçu: C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Final outputs\pre_processed_ECB_conf_final.csv


,Speaker,Date,Parsed_Text,Sentence_Rank,Parsed_Text_clean
0,Willem F. Duisenberg,1998-06-09,willem f duisenberg president of the european ...,0,willem f duisenberg european central bank 9 19...
1,Willem F. Duisenberg,1998-06-09,i may recall that the heads of state or govern...,1,recall head government appoint six ecb vicepre...
2,Willem F. Duisenberg,1998-06-09,i may also recall that on 4 june 1998 the exec...,2,recall 4 1998 releas 2 1998 decision distribut...
3,Willem F. Duisenberg,1998-06-09,in subsequent meetings of the executive board ...,3,subsequent meeting first meeting govern genera...
4,Willem F. Duisenberg,1998-06-09,the first meetings of both bodies were held ea...,4,first meeting both body held earlier



Aperçu: C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\Final outputs\pre_processed_ECB_speech_final.csv


,Speaker,Date,Parsed_Text,Sentence_Rank,Parsed_Text_clean
0,Piero Cipollone,2025-09-30,speech innovating for stability central bank m...,0,innovate stability central bank money digital ...
1,Piero Cipollone,2025-09-30,1 fast forward 12 months and the pace of trans...,1,1 fast forward 12 pace transformation quicken
2,Piero Cipollone,2025-09-30,innovative digital payment solutions and digit...,2,innovative digital payment solution digital as...
3,Piero Cipollone,2025-09-30,the opportunities and risks are being discusse...,3,opportunity risk length
4,Piero Cipollone,2025-09-30,2 so what does this mean for central banks,4,2 mean central bank


In [ ]:
# ===================== 5) DROP non-English + DROP Parsed_Text + SAVE =====================
import pandas as pd
from pathlib import Path

# --- paramètres (tu peux ajuster) ---
CONF_MIN = 0.90           # confiance minimale langue anglaise
MIN_SENT_PER_DOC = 3      # nombre min. de phrases par Doc_ID après filtrage
OUT_SUFFIX = "_en_noParsedText.csv"  # suffixe du nouveau fichier

# 1) Charger la sortie actuelle
df = pd.read_csv(r"C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_final.csv")

# 2) Supprimer colonnes d'index fantômes
drop_unnamed = [c for c in df.columns if str(c).lower().startswith("unnamed")]
df = df.drop(columns=drop_unnamed, errors="ignore")

# 3) Détection de langue locale (langid)
try:
    import langid
    # Restreindre l'espace des langues pour fiabilité/vitesse
    langid.set_languages(['en','fr','de','es','it','pt','nl'])
except Exception as e:
    raise RuntimeError(
        "Le module 'langid' est requis. Installe-le d'abord:  pip install langid"
    ) from e

# Colonne source pour la détection (préfère le texte brut si présent)
SRC_COL = "Parsed_Text_clean"

# Classifier chaque phrase -> (lang, proba)
pred = df[SRC_COL].fillna("").astype(str).apply(lambda t: langid.classify(t or ""))
df["lang"] = [p[0] for p in pred]
df["lang_prob"] = [float(p[1]) for p in pred]

# 4) Filtrer: anglais avec confiance suffisante
df = df[(df["lang"] == "en") & (df["lang_prob"] >= CONF_MIN)].copy()

# 5) Option: exclure Doc_ID trop courts après filtrage
if "Doc_ID" in df.columns and "Sentence_Rank" in df.columns:
    counts = df.groupby("Doc_ID")["Sentence_Rank"].count()
    ok_docs = counts[counts >= MIN_SENT_PER_DOC].index
    df = df[df["Doc_ID"].isin(ok_docs)]


# 7) Tri + ordre de colonnes propre
sort_cols = [c for c in ["Doc_ID","Sentence_Rank"] if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols).reset_index(drop=True)

desired = ["Doc_ID","Speaker","Date","contents","Sentence_Rank", NEW_COL]
ordered = [c for c in desired if c in df.columns]
rest = [c for c in df.columns if c not in ordered]
df = df[ordered + rest]

# 8) Sauvegarde finale (sans écraser l'original)
OUT_FINAL = Path(OUT_CSV).with_name(Path(OUT_CSV).stem + OUT_SUFFIX)
df.to_csv(OUT_FINAL, index=False, encoding="utf-8")
print(f"[OK] CSV anglais sans Parsed_Text -> {OUT_FINAL}  (lignes={len(df)}, colonnes={df.shape[1]})")



[OK] CSV anglais sans Parsed_Text -> C:\Users\Garance Latieule\Projet_ML\Analyzing-ECB\dataset\pre_processed_ECB_final_en_noParsedText.csv  (lignes=3712, colonnes=7)
